In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
import pytz
from IPython.display import VimeoVideo
from pymongo import MongoClient
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error


In [ ]:
VimeoVideo("665412117", h="c39a50bd58", width=600)


In [ ]:
host = "192.14.97.2"


In [ ]:
VimeoVideo("665412469", h="135f32c7da", width=600)


In [ ]:
client = MongoClient(host=host, port=27017)
db = client["air-quality"]
nairobi = db["nairobi"]


In [ ]:
VimeoVideo("665412480", h="c20ed3e570", width=600)


In [ ]:
def wrangle(collection):
    results = collection.find(
        {"metadata.site": 29, "metadata.measurement": "P2"},
        projection={"P2": 1, "timestamp": 1, "_id": 0},
    )

    df = pd.DataFrame(results).set_index("timestamp")

    # Localize timeZone
    df.index = df.index.tz_localize("UTC").tz_convert("Africa/Nairobi")

    # Remove outliers
    df = df[df["P2"] < 500]

    # Resample to 1H window , fill missing value
    df = df["P2"].resample("1H").mean().fillna(method="ffill").to_frame()

    # Add lag feature
    df["P2.L1"] = df["P2"].shift(1)
    
    # Drop NaN rows
    df.dropna(inplace=True)
    
    return df


In [ ]:
VimeoVideo("665412496", h="d757475f7c", width=600)


In [ ]:
df = wrangle(nairobi)
print(df.shape)
df.head()


In [ ]:
# Check your work
assert any([isinstance(df, pd.DataFrame), isinstance(df, pd.Series)])
assert len(df) <= 32907
assert isinstance(df.index, pd.DatetimeIndex)


In [ ]:
VimeoVideo("665412520", h="e03eefff07", width=600)


In [ ]:
# Check your work
assert df.index.tzinfo == pytz.timezone("Africa/Nairobi")


In [ ]:
VimeoVideo("665412546", h="97792cb982", width=600)


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
df["P2"].plot(kind="box",vert=False, title="Distribution of PM2.5 Reading", ax=ax)


In [ ]:
VimeoVideo("665412573", h="b46049021b", width=600)


In [ ]:
# Check your work
assert len(df) <= 32906


In [ ]:
VimeoVideo("665412594", h="e56c2f6839", width=600)


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
df["P2"].plot(kind="line", xlabel="Time", ylabel="PM2.5", title="PM2.5 time series", ax=ax )


In [ ]:
VimeoVideo("665412601", h="a16c5a73fc", width=600)


In [ ]:
# Check your work
assert len(df) <= 2928


In [ ]:
VimeoVideo("665412649", h="d2e99d2e75", width=600)


In [ ]:
fig, ax = plt.subplots(figsize=(15, 6))
df["P2"].rolling(168).mean().plot(ax=ax,ylabel= "PM2.5", title="weekly rolling Average")


In [ ]:
VimeoVideo("665412693", h="c3bca16aff", width=600)


In [ ]:
# Check your work
assert len(df) <= 11686
assert df.shape[1] == 2


In [ ]:
VimeoVideo("665412732", h="059e4088c5", width=600)


In [ ]:
df.corr()


In [ ]:
VimeoVideo("665412741", h="7439cb107c", width=600)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(x=df["P2.L1"],y=df["P2"])
ax.plot([0,120], [0,120], linestyle="--", color="orange")
plt.xlabel("P2.L1")
plt.ylabel("P2")
plt.title("PM2.5 Autocorrelation");


In [ ]:
VimeoVideo("665412762", h="a5eba496f7", width=600)


In [ ]:
target = "P2"
y = df[target]
X = df.drop(columns=target)


In [ ]:
VimeoVideo("665412785", h="03118eda71", width=600)


In [ ]:
cutoff = int(len(X) * 0.8)
X_train, y_train = X.iloc[:cutoff], y.iloc[:cutoff]
X_test, y_test = X.iloc[cutoff:], y.iloc[cutoff:]


In [ ]:
y_pred_baseline = [y_train.mean()] * len(y_train)
mae_baseline = mean_absolute_error(y_train,y_pred_baseline)

print("Mean P2 Reading:", round(y_train.mean(),2))
print("Baseline MAE:", round(mae_baseline, 2))


In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)


In [ ]:
VimeoVideo("665412844", h="129865775d", width=600)


In [ ]:
training_mae = mean_absolute_error(y_train, model.predict(X_train))
test_mae = mean_absolute_error(y_test, model.predict(X_test))
print("Training MAE:",round(training_mae, 2))
print("Test MAE:", round(test_mae, 2))


In [ ]:
intercept = model.intercept_.round(2)
coefficient = model.coef_.round(2)

print(f"P2 = {intercept} + ({coefficient} * P2.L1)")


In [ ]:
VimeoVideo("665412870", h="318d69683e", width=600)


In [ ]:
df_pred_test = pd.DataFrame(
    {
        "y_test": y_test,
        "y_pred": model.predict(X_test)
    }
)
df_pred_test.head()


In [ ]:
VimeoVideo("665412891", h="39d7356a26", width=600)


In [ ]:
fig = px.line(df_pred_test, labels={"value":"P2"})
fig.show()
